# Diabetic Encounters: Relational Schema Design

**Source:** `data/processed/diabetic_data_cleaned.csv` — output of 
`01_diabetic_data_wrangling.ipynb` (101,766 encounters, 55 fields).

**Purpose:** Verify the cleaned dataset's assumptions hold on a fresh load, 
then design a normalized MySQL schema (`sql/schema.sql`) to support the 
clinical decision-support dashboard.

This notebook does two things:
1. Confirms the saved CSV round-trips correctly (right shape, right types) — 
   the first real test of the file produced by the previous notebook.
2. Investigates one open question before finalizing table boundaries: 
   whether `race` and `gender` are stable per patient, which determines 
   whether they belong in a `patients` table or must stay in `encounters`.

## Load and Verify

Loading from the saved CSV, not a running kernel — the first real check that 
`01_diabetic_data_wrangling.ipynb`'s output is actually usable downstream.

In [1]:
import pandas as pd
diabetic = pd.read_csv('../data/processed/diabetic_data_cleaned.csv')
print(diabetic.shape)
diabetic.dtypes

(101766, 56)


encounter_id                  int64
patient_nbr                   int64
race                            str
gender                          str
age                             str
admission_type_id             int64
discharge_disposition_id      int64
admission_source_id           int64
time_in_hospital              int64
payer_code                      str
medical_specialty               str
num_lab_procedures            int64
num_procedures                int64
num_medications               int64
number_outpatient             int64
number_emergency              int64
number_inpatient              int64
diag_1                          str
diag_2                          str
diag_3                          str
number_diagnoses              int64
max_glu_serum                   str
A1Cresult                       str
metformin                       str
repaglinide                     str
nateglinide                     str
chlorpropamide                  str
glimepiride                 

## ETL Prototype: Split cleaned data into schema-matching tables

Before writing `etl/clean_diabetes_data.py`, prototype the split-and-load 
logic here — three pieces matching the schema (`patients`, `encounters`, 
`encounter_medications`), plus renaming the two columns that don't match 
MySQL-safe names (`change` → `change_flag`, `A1Cresult` → `a1c_result`).

In [2]:
diabetic = diabetic.rename(columns={
    'change': 'change_flag',
    'A1Cresult': 'a1c_result',
    'diabetesMed': 'diabetes_med',
    'glyburide-metformin': 'glyburide_metformin',
    'glipizide-metformin': 'glipizide_metformin',
    'glimepiride-pioglitazone': 'glimepiride_pioglitazone',
    'metformin-rosiglitazone': 'metformin_rosiglitazone',
    'metformin-pioglitazone': 'metformin_pioglitazone'
})

assert 'change_flag' in diabetic.columns
assert 'a1c_result' in diabetic.columns
assert 'diabetes_med' in diabetic.columns
assert 'glyburide_metformin' in diabetic.columns
print("All renamed columns confirmed present")

All renamed columns confirmed present


### Split into three tables matching the schema
`patients` (unique IDs only), `encounters` (one row per admission), and 
`encounter_medications` (drug-level detail, joined back via `encounter_id`).

In [3]:
# patients — deliberately minimal, per the schema decision
patients_df = diabetic[['patient_nbr']].drop_duplicates().reset_index(drop=True)

# encounters — everything except the 23 medication columns
medication_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide_metformin',
    'glipizide_metformin', 'glimepiride_pioglitazone',
    'metformin_rosiglitazone', 'metformin_pioglitazone'
]

encounters_df = diabetic.drop(columns=medication_cols)

# encounter_medications — encounter_id + the 23 medication columns
medications_df = diabetic[['encounter_id'] + medication_cols]

print("patients_df:", patients_df.shape)
print("encounters_df:", encounters_df.shape)
print("medications_df:", medications_df.shape)

assert patients_df.shape[0] == diabetic['patient_nbr'].nunique()
assert encounters_df.shape[0] == diabetic.shape[0]
assert medications_df.shape[0] == diabetic.shape[0]
print("Row counts verified against source")

patients_df: (71518, 1)
encounters_df: (101766, 33)
medications_df: (101766, 24)
Row counts verified against source


### Connect to MySQL and load the three tables
Loading order matters: `patients` and the three lookup tables first (since 
`encounters` has foreign keys referencing them), then `encounters`, then 
`encounter_medications` last (since it references `encounters`).

In [4]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv('../.env')

db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_name = os.getenv('DB_NAME')

engine = create_engine(f'mysql+mysqlconnector://{db_user}:{db_password}@{db_host}/{db_name}')

# Quick connection test
with engine.connect() as conn:
    print("Connected successfully")

Connected successfully


In [5]:
encounters_df = encounters_df.drop(columns=['admission_type_desc', 'discharge_disposition_desc', 'admission_source_desc'])
print(encounters_df.shape)

(101766, 30)


In [12]:
from sqlalchemy import inspect
inspector = inspect(engine)

for df, table_name in [(patients_df, 'patients'), (encounters_df, 'encounters'), (medications_df, 'encounter_medications')]:
    table_cols = set(col['name'] for col in inspector.get_columns(table_name))
    df_cols = set(df.columns)
    print(f"{table_name}: extra_in_df={df_cols - table_cols}, missing_from_df={table_cols - df_cols}")

patients: extra_in_df=set(), missing_from_df=set()
encounters: extra_in_df=set(), missing_from_df=set()
encounter_medications: extra_in_df=set(), missing_from_df=set()


In [14]:
admission_type = pd.read_csv('../data/raw/admission_type.csv', keep_default_na=False, na_values=[''])
discharge_disposition = pd.read_csv('../data/raw/discharge_disposition.csv', keep_default_na=False, na_values=[''])
admission_source = pd.read_csv('../data/raw/admission_source.csv', keep_default_na=False, na_values=[''])

print(admission_type.shape, discharge_disposition.shape, admission_source.shape)

(8, 2) (30, 2) (25, 2)


In [15]:
admission_type.to_sql('admission_type', engine, if_exists='append', index=False)
discharge_disposition.to_sql('discharge_disposition', engine, if_exists='append', index=False)
admission_source.to_sql('admission_source', engine, if_exists='append', index=False)
print("Lookup tables loaded")

Lookup tables loaded


In [16]:
patients_df.to_sql('patients', engine, if_exists='append', index=False)
print(f"Loaded {len(patients_df):,} patients")

Loaded 71,518 patients


In [19]:
encounters_df.to_sql('encounters', engine, if_exists='append', index=False)
print(f"Loaded {len(encounters_df):,} encounters")

Loaded 101,766 encounters


In [22]:
medications_df.to_sql('encounter_medications', engine, if_exists='append', index=False)
print(f"Loaded {len(medications_df):,} medication records")

Loaded 101,766 medication records
